In [1]:
import librosa
import IPython.display as ipd
import requests
import io
import re

from google.cloud import storage

import soundfile as sf

# Note
Mixer functions originally built for wav files but some of these are stored as flac / other types - may need to change mixer functions to sf.read() instead of wavfile.read()

# Step 1
Get URLs from event buckets of interest and store them in a dictionary that has a user-friendly key for the clip.
I need to expand this code to iterate through all folders with sound clips and not just this one but this is the idea:

In [3]:
bucket_name = "noaa-passive-bioacoustic"
prefix = "sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/"

client = storage.Client.create_anonymous_client()
bucket = client.bucket(bucket_name)
blobs = client.list_blobs(bucket, prefix=prefix)

urls = {}

for blob in blobs:  
    if not blob.name.endswith('.wav'):
        continue

    match = re.search(r"SanctSound_OC03_02_([^_]+)_\d{8}T\d{6}Z\.wav$", blob.name)
    if not match:
        continue

    event_name = match.group(1)

    if "whale" in event_name:
        species = event_name.split("whale") 
        formatted_name = ' '.join([p.capitalize() for p in species if p] + ['whale'])
    else:
        formatted_name = event_name.capitalize()

    formatted_name += " - SanctSound"

    urls[formatted_name] = bucket_name + "/" + blob.name


urls

{'Humpback whale - SanctSound': 'noaa-passive-bioacoustic/sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/SanctSound_OC03_02_humpbackwhale_20191104T151853Z.wav',
 'Killer whale - SanctSound': 'noaa-passive-bioacoustic/sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/SanctSound_OC03_02_killerwhale_20191128T091339Z.wav',
 'Ship - SanctSound': 'noaa-passive-bioacoustic/sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/SanctSound_OC03_02_ship_20200228T080107Z.wav',
 'Soundscape - SanctSound': 'noaa-passive-bioacoustic/sanctsound/products/sound_clips/oc03/sanctsound_oc03_02_sound_clips/data/SanctSound_OC03_02_soundscape_20191220T211558Z.wav'}

# Step 2
Function that will load audio for a given key.
My thought is user could "add" individual files they want to use to their library since it would probably take too long to store all in local memory (even though they'd be stored numerically)

In [66]:
def load_audio(key):

    if key not in urls:
        raise(ValueError(f"Key not found."))

    full_url = f"https://storage.googleapis.com/{urls[key]}"
    response = requests.get(full_url)
    response.raise_for_status()  # raise error if request failed
    audio_bytes = io.BytesIO(response.content)
    
    # Read audio
    data, sr = sf.read(audio_bytes)
    
    return data, sr

load_audio("Killer whale - SanctSound")

(array([ 0.00000000e+00,  3.05175781e-05, -6.10351562e-05, ...,
        -1.83105469e-04,  4.27246094e-04, -3.35693359e-04],
       shape=(2880000,)),
 48000)